In [1]:
import os

In [2]:
%pwd

'c:\\projects\\SellWise\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\projects\\SellWise'

In [5]:
import pandas as pd
import os

In [6]:
# Inspect one of the generated preprocessed grid files
grid_part_1 = pd.read_pickle("artifacts/data_ingestion/processed/grid_part_1.pkl")
grid_part_1.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,release
0,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,1,12.0,0
1,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,1,2.0,0
2,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0,0
3,HOBBIES_1_012_CA_1_evaluation,HOBBIES_1_012,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0,0
4,HOBBIES_1_015_CA_1_evaluation,HOBBIES_1_015,HOBBIES_1,HOBBIES,CA_1,CA,1,4.0,0


In [7]:
grid_part_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47735397 entries, 0 to 47735396
Data columns (total 9 columns):
 #   Column    Dtype   
---  ------    -----   
 0   id        category
 1   item_id   category
 2   dept_id   category
 3   cat_id    category
 4   store_id  category
 5   state_id  category
 6   d         int16   
 7   sales     float64 
 8   release   int16   
dtypes: category(6), float64(1), int16(2)
memory usage: 910.7 MB


In [8]:
grid_part_1.shape

(47735397, 9)

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    data_path: Path
    STATUS_FILE: Path
    all_schema: dict  # Loaded from schema.yaml COLUMNS

In [10]:
from SellWise.constants import *
from SellWise.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS  # Loads all expected features from schema.yaml

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            STATUS_FILE=Path(config.STATUS_FILE),
            all_schema=schema
        )

        return data_validation_config

In [12]:
import os
from SellWise import logger

In [13]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_and_columns(self) -> bool:
        try:
            validation_status = True
            
            # 1. Check artifact existence & non-zero size
            required_files = [
                "grid_part_1.pkl",
                "grid_part_2.pkl",
                "grid_part_3.pkl",
                "lags_df_28.pkl",
                "mean_encoding_df.pkl"
            ]

            existing_files = os.listdir(self.config.data_path)

            for file in required_files:
                file_full_path = os.path.join(self.config.data_path, file)
                if file not in existing_files or os.path.getsize(file_full_path) == 0:
                    validation_status = False
                    logger.error(f"Missing or empty artifact: {file}")
                    break

            # 2. Check column schema on sample artifact (grid_part_1.pkl)
            if validation_status:
                sample_data = pd.read_pickle(os.path.join(self.config.data_path, "grid_part_1.pkl"))
                all_cols = list(sample_data.columns)
                expected_schema = self.config.all_schema.keys()

                for col in all_cols:
                    if col not in expected_schema:
                        validation_status = False
                        logger.error(f"Unexpected column found: {col}")
                        break

            # 3. Write overall status to status.txt
            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [14]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_files_and_columns()
except Exception as e:
    raise e

[2026-09-23 00:12:25,648: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-23 00:12:25,659: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-23 00:12:25,668: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-23 00:12:25,670: INFO: common: created directory at: artifacts]
[2026-09-23 00:12:25,672: INFO: common: created directory at: artifacts/data_validation]
